# BrowseComp-Plus Retrieval — Client Test Notebook

Exercises the service end to end: connection, response schema, auth, error handling,
latency, and an optional recall check against the benchmark qrels.

* Sections 1–10 — `POST /retrieve` (the Search-R1 `{document: {title, text}}` shape).
* Section 11 — the MCP tool-parity endpoints `POST /search` and `GET /get_document`.
* Section 12 — fetching one document in full by its docid.

The service pulls **100 candidates from each leg** (BM25 + dense), fuses them with RRF,
and reranks every fused candidate down to the final **top 10**.

Run top to bottom. Each cell is independent of the notebook's own state except where noted.

## 1. Configuration

`BASE_URL` and `TOKEN` are read from the environment so the notebook carries no secrets.
Inside this container the defaults resolve automatically.

In [5]:
import os

# Inside the container: Caddy's auth edge is on port 10100.
# From outside: http://$PUBLIC_IPADDR:$VAST_TCP_PORT_10100
BASE_URL = os.environ.get("BCP_RETRIEVAL_URL", "http://127.0.0.1:10100")
TOKEN = os.environ.get("BCP_RETRIEVAL_TOKEN") or os.environ.get("OPEN_BUTTON_TOKEN", "")

# Reranking ~200 long documents on one GPU takes ~22s, and concurrent requests
# queue behind a server-side lock. Keep the timeout generous.
TIMEOUT = 180

print(f"BASE_URL : {BASE_URL}")
print(f"TOKEN    : {'set (' + str(len(TOKEN)) + ' chars)' if TOKEN else 'MISSING'}")
assert TOKEN, "No token found. Set BCP_RETRIEVAL_TOKEN or OPEN_BUTTON_TOKEN."

BASE_URL : http://127.0.0.1:10100
TOKEN    : set (64 chars)


## 2. The client

A session with the auth header attached once. `retrieve()` returns the raw hit list;
`clean_text()` strips the residual YAML front-matter the API documents as a known artifact.

In [6]:
import re
import time

import requests

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {TOKEN}"})

MAX_FRONT_MATTER_LINES = 40


def retrieve(query: str, timeout: int = TIMEOUT) -> list[dict]:
    """Return up to 10 hits as flat dicts: {docid, title, text}."""
    response = session.post(
        f"{BASE_URL}/retrieve", json={"query": query}, timeout=timeout
    )
    response.raise_for_status()
    return [
        {"docid": item["docid"], **item["document"]}
        for item in response.json().get("result", [])
    ]


def clean_text(text: str) -> str:
    """Strip residual YAML front-matter left by the server's title extraction.

    The corpus mixes several shapes -- `---` fences, bare `key: value` runs, and
    documents where both appear stacked -- so peel leading blank lines, fences
    and key lines repeatedly until the text stops changing. Stopping at the first
    fence is not enough: some documents nest a second block behind it.
    """
    lines = text.split("\n")
    i, changed = 0, True
    while changed and i < min(len(lines), MAX_FRONT_MATTER_LINES):
        changed = False
        while i < len(lines) and not lines[i].strip():
            i, changed = i + 1, True
        if i < len(lines) and lines[i].strip() == "---":
            i, changed = i + 1, True
        while i < len(lines) and re.match(r"^\s*[\w.-]+:\s", lines[i]):
            i, changed = i + 1, True
    return "\n".join(lines[i:]).strip()


def show(hits: list[dict], width: int = 70, snippet: int = 0) -> None:
    if not hits:
        print("(no results)")
        return
    for rank, hit in enumerate(hits, 1):
        print(f"{rank:>3}. [{hit['docid']:>6}] {hit['title'][:width]}")
        if snippet:
            body = clean_text(hit["text"])[:snippet].replace("\n", " ")
            print(f"      {body}...")


print("client ready")

client ready


## 3. Liveness check

If this fails, the service is probably still loading (model + index take 2–3 minutes):
`supervisorctl status bcp-retrieval`.

In [7]:
t0 = time.time()
try:
    hits = retrieve("health check")
    print(f"Healthy — {len(hits)} results in {time.time() - t0:.1f}s")
except requests.exceptions.HTTPError as e:
    print(f"Unhealthy — HTTP {e.response.status_code}: {e.response.text[:200]}")
except requests.exceptions.RequestException as e:
    print(f"Unhealthy — {type(e).__name__}: {e}")

Healthy — 10 results in 25.7s


## 4. A single query

The canonical example from the API guide. Expect docid `16659`
("1870s – 1940s: Telephone") at or near the top.

In [ ]:
t0 = time.time()
hits = retrieve("who invented the telephone")
print(f"{len(hits)} results in {time.time() - t0:.1f}s\n")
show(hits, snippet=110)

10 results in 21.9s

  1. [ 16659] 1870s – 1940s: Telephone
      This timeline is provided to help show how the dominant form of communication changes as rapidly as innovators...
  2. [ 76829] This Month in Business History: AT&T Monopoly & Breakup of the Bell Sy
      Forty years ago this month, the federal government settled a lawsuit with American Telephone & Telegraph Compa...
  3. [ 22293] The Franklin School
      A historic discovery  In 1880, inventor Alexander Graham Bell made his first successful transmission of a "pho...
  4. [  9966] Alfred Vail
      Alfred Vail  Inventor  Alfred Lewis Vail (September 25, 1807 – January 18, 1859) was an American machinist and...
  5. [ 54451] Young Edison
      THOMAS EDISON AND MENLO PARK  More than any other inventor in history, Thomas Edison is responsible for the te...
  6. [ 68884] Western Electric - Wikipedia
      Western Electric Co., Inc. was an American electrical engineering and manufacturing company that operated from...
  7. 

## 5. Response schema

Confirms the documented contract: items are `{"document": {...}, "docid": ...}`,
accessed by key rather than position. Note the raw `text` still carries front-matter.

In [ ]:
import json

raw = session.post(
    f"{BASE_URL}/retrieve", json={"query": "who invented the telephone"}, timeout=TIMEOUT
).json()

first = raw["result"][0]
print("top-level keys :", list(raw.keys()))
print("item keys      :", list(first.keys()))
print("document keys  :", list(first["document"].keys()))
print("scores exposed :", "score" in first or "score" in first["document"])
print()
print(json.dumps({**first, "document": {**first["document"], "text": first["document"]["text"][:160] + "..."}}, indent=2)[:700])

top-level keys : ['result']
item keys      : ['document', 'docid']
document keys  : ['title', 'text']
scores exposed : False

{
  "document": {
    "title": "1870s \u2013 1940s: Telephone",
    "text": "date: 2025-01-01\n---\nThis timeline is provided to help show how the dominant form of communication changes as rapidly as innovators develop new technologies.\n\nA..."
  },
  "docid": "16659"
}


## 6. Contract checks

Auth enforcement, schema validation, the ignored `k` override, and snippet truncation.

In [ ]:
def check(label, expected, actual):
    ok = "PASS" if expected == actual else "FAIL"
    print(f"  [{ok}] {label:<44} expected={expected!s:<8} got={actual}")


print("auth")
check(
    "no token is rejected",
    401,
    requests.post(f"{BASE_URL}/retrieve", json={"query": "x"}, timeout=30).status_code,
)
check(
    "bad token is rejected",
    401,
    requests.post(
        f"{BASE_URL}/retrieve",
        json={"query": "x"},
        headers={"Authorization": "Bearer nope"},
        timeout=30,
    ).status_code,
)
check(
    "?token= query-string auth works",
    200,
    requests.post(
        f"{BASE_URL}/retrieve?token={TOKEN}", json={"query": "telephone"}, timeout=TIMEOUT
    ).status_code,
)

print("\nrequest validation")
check(
    "missing 'query' key -> 422",
    422,
    session.post(f"{BASE_URL}/retrieve", json={"q": "wrong"}, timeout=30).status_code,
)

print("\nfixed top-k and truncation")
overridden = session.post(
    f"{BASE_URL}/retrieve", json={"query": "telephone", "k": 3}, timeout=TIMEOUT
).json()["result"]
check("custom 'k' is silently ignored", 10, len(overridden))
longest = max(len(h["document"]["text"]) for h in overridden)
print(f"  [INFO] longest snippet: {longest} chars (512-token cap, ~2,300 chars)")

auth
  [PASS] no token is rejected                         expected=401      got=401
  [PASS] bad token is rejected                        expected=401      got=401


  [PASS] ?token= query-string auth works              expected=200      got=200

request validation
  [PASS] missing 'query' key -> 422                   expected=422      got=422

fixed top-k and truncation
  [PASS] custom 'k' is silently ignored               expected=10       got=10
  [INFO] longest snippet: 2508 chars (512-token cap, ~2,300 chars)


## 7. Latency across several queries

Every request reranks ~200 long documents on a single GPU, so expect ~20s and treat
the endpoint as high-latency: cache by `docid`, and never fan out in parallel expecting
a speedup — requests serialize behind a server-side lock.

In [ ]:
probe_queries = [
    "who invented the telephone",
    "Alexander Graham Bell telephone patent dispute",
    "university research group founded in 2009",
    "three-day academic conference held in 2002",
]

timings = []
for q in probe_queries:
    t0 = time.time()
    hits = retrieve(q)
    dt = time.time() - t0
    timings.append(dt)
    top = f"[{hits[0]['docid']}] {hits[0]['title'][:46]}" if hits else "(empty)"
    print(f"{dt:6.1f}s  n={len(hits):<3} {q[:44]:<46} -> {top}")

print(f"\nmean {sum(timings) / len(timings):.1f}s   min {min(timings):.1f}s   max {max(timings):.1f}s")

  22.1s  n=10  who invented the telephone                     -> [16659] 1870s – 1940s: Telephone
  27.3s  n=10  Alexander Graham Bell telephone patent dispu   -> [50937] The Breakup of "Ma Bell
  26.6s  n=10  university research group founded in 2009      -> [68484] ABOUT US
  28.6s  n=10  three-day academic conference held in 2002     -> [79858] Professor Jiaxing HONG invited to speak at ICM

mean 26.2s   min 22.1s   max 28.6s


## 8. Concurrency behaviour

Three requests fired at once. Total wall time should be roughly the sum of the
individual times, not the max — that is the GPU lock doing its job.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

burst = ["telephone invention", "railway bridge collapse", "antarctic expedition 1911"]

t0 = time.time()
with ThreadPoolExecutor(max_workers=len(burst)) as pool:
    results = list(pool.map(retrieve, burst))
elapsed = time.time() - t0

for q, hits in zip(burst, results):
    print(f"  n={len(hits):<3} {q}")
print(f"\n{len(burst)} concurrent requests in {elapsed:.1f}s "
      f"(~{elapsed / len(burst):.1f}s each — serialized, not parallel)")

  n=10  telephone invention
  n=10  railway bridge collapse
  n=10  antarctic expedition 1911

3 concurrent requests in 79.3s (~26.4s each — serialized, not parallel)


## 9. Optional: recall against the benchmark qrels

Measures how often the gold documents land in the returned top 10. Requires the
decrypted queries (`data/queries.tsv`, produced by `scripts_build_index/decrypt_dataset.py`).

At ~20s per query this is slow — keep `N_QUERIES` small. For proper pipeline
evaluation use `scripts_evaluation/tune_pipeline.py`, which scores offline and sweeps
several configurations in one pass.

In [ ]:
import collections
import pathlib
import random

N_QUERIES = 5
REPO = pathlib.Path("/workspace/BrowseComp-Plus")
queries_path = REPO / "data/queries.tsv"
qrels_path = REPO / "topics-qrels/qrel_golds.txt"

if not queries_path.exists():
    print(f"{queries_path} not found — skipping.")
    print("Generate it with:")
    print("  python scripts_build_index/decrypt_dataset.py \\")
    print("      --output data/decrypted.jsonl --generate-tsv data/queries.tsv")
else:
    queries = {}
    for line in queries_path.read_text().splitlines():
        qid, _, text = line.partition("\t")
        if text:
            queries[qid] = text

    qrels = collections.defaultdict(set)
    for line in qrels_path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 4 and int(parts[3]) > 0:
            qrels[parts[0]].add(parts[2])

    qids = sorted(set(queries) & set(qrels))
    random.Random(0).shuffle(qids)
    qids = qids[:N_QUERIES]
    print(f"{len(queries)} queries available; evaluating {len(qids)}\n")

    recalls = []
    for qid in qids:
        gold = qrels[qid]
        got = {h["docid"] for h in retrieve(queries[qid])}
        r = len(got & gold) / len(gold)
        recalls.append(r)
        print(f"  qid={qid:<6} gold={len(gold):<3} hit={len(got & gold):<3} recall@10={r:.2f}")
        print(f"    {queries[qid][:96]}...")

    print(f"\nmean recall@10 over {len(recalls)} queries: {sum(recalls) / len(recalls):.4f}")
    print("(reference: ~0.39 measured over 40 queries — expect wide variance at this sample size)")

830 queries available; evaluating 5



  qid=746    gold=1   hit=1   recall@10=1.00
    A former African football player born in the late 1970s married a model in the mid-2010s. He was...
  qid=282    gold=1   hit=1   recall@10=1.00
    There is a show that was released between the years 1940 and 2000. In this show, there is a char...
  qid=1038   gold=1   hit=1   recall@10=1.00
    Somewhere between 2001 and 2013, the yearly meeting of a club was announced in a newsletter, at ...
  qid=413    gold=8   hit=3   recall@10=0.38
    This person won a gold medal at the Olympics in a year between 2011 and 2023 inclusive, in a spo...
  qid=734    gold=2   hit=0   recall@10=0.00
    An actress born in a city whose flag bears a legendary creature was seen in a short mini-series ...

mean recall@10 over 5 queries: 0.6750
(reference: ~0.39 measured over 40 queries — expect wide variance at this sample size)


## 10. Using it as an LLM tool

The tool definition for an agent loop, plus a formatter that turns hits into a
citable context block. Docids are stable, so they make good citation anchors.

In [ ]:
SEARCH_TOOL = {
    "name": "search_browsecomp_plus",
    "description": (
        "Searches the BrowseComp-Plus web corpus. Returns up to 10 ranked document "
        "snippets (max 512 tokens each) with docids and titles. The corpus is a fixed "
        "historical snapshot without date/site filtering or pagination capabilities. "
        "Re-phrase queries if initial search fails to produce desired details."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Natural language search query."}
        },
        "required": ["query"],
    },
}


def as_context(query: str, max_chars: int = 600) -> str:
    """Render hits as a numbered, citable block for an LLM prompt."""
    blocks = []
    for hit in retrieve(query):
        body = clean_text(hit["text"])[:max_chars]
        blocks.append(f"[{hit['docid']}] {hit['title']}\n{body}")
    return "\n\n".join(blocks)


context = as_context("who invented the telephone", max_chars=220)
print(f"context block: {len(context)} chars\n")
print(context[:900])

context block: 2752 chars

[16659] 1870s – 1940s: Telephone
This timeline is provided to help show how the dominant form of communication changes as rapidly as innovators develop new technologies.

A brief historical overview: The printing press was the big innovation in communic

[76829] This Month in Business History: AT&T Monopoly & Breakup of the Bell System
Forty years ago this month, the federal government settled a lawsuit with American Telephone & Telegraph Company (AT&T), prompting the breakup of one of the largest and most powerful monopolies of the 19th and 20th centu

[22293] The Franklin School
A historic discovery

In 1880, inventor Alexander Graham Bell made his first successful transmission of a "photophone" message from the rooftop of the Franklin School to his lab on L Street. The photophone was a vital pr

[9966] Alfred Vail
Alfred Vail

Inventor

Alfred Lewis Vail (September 25, 1807 – January 


## 11. Tool-parity endpoints: `/search` and `/get_document`

These two are the HTTP equivalents of the MCP tools in `searcher/tools.py` — same
searcher calls, same payloads — so an agent can point at either transport without
reshaping its tool handlers.

* `POST /search` → a bare array of `{docid, score, snippet}`. Same ranking as
  `/retrieve` and the same ~25s cost, but it keeps the re-ranker score and leaves the
  snippet raw (front-matter intact) instead of splitting title/text.
* `GET /get_document?docid=…` → `{docid, text}`, the **full untruncated** document.
  Served from the in-memory corpus map: no GPU, no lock, milliseconds. Fan it out
  freely after a search — that is the intended pattern.

In [ ]:
def search(query: str, timeout: int = TIMEOUT) -> list[dict]:
    """MCP-parity `search`: [{docid, score, snippet}], best first."""
    response = session.post(f"{BASE_URL}/search", json={"query": query}, timeout=timeout)
    response.raise_for_status()
    return response.json()


def get_document(docid: str, timeout: int = 30) -> dict | None:
    """MCP-parity `get_document`: {docid, text} in full, or None if the docid is unknown."""
    response = session.get(
        f"{BASE_URL}/get_document", params={"docid": docid}, timeout=timeout
    )
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()


t0 = time.time()
hits = search("who invented the telephone")
print(f"/search — {len(hits)} hits in {time.time() - t0:.1f}s, keys {list(hits[0])}\n")
for hit in hits[:3]:
    head = clean_text(hit["snippet"])[:58].replace("\n", " ")
    print(f"  [{hit['docid']:>6}] score={hit['score']:>7.4f}  {head}...")

top = hits[0]
t0 = time.time()
doc = get_document(top["docid"])
print(f"\n/get_document — docid={top['docid']} in {(time.time() - t0) * 1000:.0f}ms")
print(f"  snippet {len(top['snippet']):>6} chars (512-token cap)")
print(f"  full    {len(doc['text']):>6} chars (untruncated)")
print(f"  snippet is the head of the full document: {doc['text'].startswith(top['snippet'][:300])}")
print(f"\n  unknown docid -> {get_document('no-such-doc')}")

## 12. Fetching a single document by docid

`GET /get_document` is a plain GET with the docid in the query string — no JSON body,
no search involved. It reads the server's in-memory corpus map, so it costs
milliseconds, never takes the GPU lock, and returns the document **in full** instead of
the 512-token snippet.

```
GET {BASE_URL}/get_document?docid=16659      ->  {"docid": "16659", "text": "..."}
```

Docids come from `/search` or `/retrieve` — this endpoint looks up an id, it cannot
find a document by title or content.

In [ ]:
DOCID = "16659"  # any docid returned by /search or /retrieve

t0 = time.time()
response = session.get(f"{BASE_URL}/get_document", params={"docid": DOCID}, timeout=30)
print(f"GET {response.url}")
print(f"    HTTP {response.status_code} in {(time.time() - t0) * 1000:.0f}ms\n")

doc = response.json()
print(f"keys  : {list(doc)}")
print(f"docid : {doc['docid']}")
print(f"text  : {len(doc['text'])} chars — full document, no 512-token cap\n")
print(clean_text(doc["text"])[:400].replace("\n", " "), "...")

### Auth variants and the missing-docid case

The docid already occupies the query string, so query-string auth is appended with `&`
rather than `?`. An unknown docid comes back as a `404` with a JSON `detail`, not as an
empty document — so a client should branch on the status code, as `get_document()` in
section 11 does.

In [ ]:
def check_status(label, expected, actual):
    ok = "PASS" if expected == actual else "FAIL"
    print(f"  [{ok}] {label:<38} expected={expected!s:<8} got={actual}")


print("auth")
check_status(
    "bearer header (the session default)",
    200,
    session.get(f"{BASE_URL}/get_document", params={"docid": DOCID}, timeout=30).status_code,
)
check_status(
    "?docid=...&token=... query string",
    200,
    requests.get(f"{BASE_URL}/get_document?docid={DOCID}&token={TOKEN}", timeout=30).status_code,
)
check_status(
    "no token is rejected",
    401,
    requests.get(f"{BASE_URL}/get_document", params={"docid": DOCID}, timeout=30).status_code,
)

print("\nerror cases")
missing = session.get(f"{BASE_URL}/get_document", params={"docid": "no-such-doc"}, timeout=30)
check_status("unknown docid -> 404", 404, missing.status_code)
print(f"         body: {missing.json()}")
check_status(
    "omitted docid -> 422",
    422,
    session.get(f"{BASE_URL}/get_document", timeout=30).status_code,
)

### The intended pattern: one search, then fan out by docid

Search once (~25s, GPU-bound and serialized), then pull the documents worth reading in
full. Unlike search, these fetches take no lock, so they can be issued in parallel and
cached by docid — the snippet is only the head of the document, and the answer is often
further down.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

t0 = time.time()
hits = search("who invented the telephone")
t_search = time.time() - t0

t0 = time.time()
with ThreadPoolExecutor(max_workers=3) as pool:
    docs = list(pool.map(lambda hit: get_document(hit["docid"]), hits[:3]))
t_fetch = time.time() - t0

print(f"1x search    {t_search:6.1f}s   {len(hits)} hits")
print(f"3x fetch     {t_fetch:6.2f}s   (parallel — no GPU lock)\n")
for hit, full in zip(hits[:3], docs):
    print(f"  [{hit['docid']:>6}] snippet {len(hit['snippet']):>6} chars"
          f"  ->  full {len(full['text']):>6} chars")